In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import RidgeClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score
import optuna

scaler = MinMaxScaler()
df_courses_tasks = pd.read_csv("train/courses_tasks_train.csv")
df_activity_log = pd.read_csv("train/activity_log_train.csv")
df_students = pd.read_csv("train/students_train.csv")
df_task_marks = pd.read_csv("train/task_marks_train.csv")
df_labels = pd.read_csv("train/final_marks_train.csv")

In [3]:
# NEW - Breadth and intensity of study/engagement a student has with non graded tasks.
def course_task_non_grade_revised(df_courses_tasks, df_activity_log):
    """
    Computes student-level features based on engagement with non-graded tasks,
    but aggregated per course to maintain context.
    - non_graded_interactions_per_course: intensity of study
    - unique_non_graded_tasks_viewed_per_course: breadth of study
    
    Returns a dataframe with features for each student-course pair.
    """
    # 1. Identify all non-graded tasks across the entire dataset.
    # A task is non-graded if `is_resource` is True or if its `weight` is 0.
    non_graded_tasks = df_courses_tasks[
        (df_courses_tasks["is_resource"] == True) | (df_courses_tasks["weight"] == 0)
    ]
    
    # 2. Filter the activity log to only include interactions with these specific non-graded tasks.
    logs_non_graded = df_activity_log[
        df_activity_log["task_id"].isin(non_graded_tasks["task_id"])
    ].copy() # Use .copy() to avoid SettingWithCopyWarning
    
    if logs_non_graded.empty:
        # If no interactions with non-graded tasks, return an empty DataFrame with the correct columns.
        return pd.DataFrame(columns=[
            "student_id",
            "course_id",
            "non_graded_interactions_per_course",
            "unique_non_graded_tasks_viewed_per_course"
        ])
    
    # 3. Group by both `student_id` and `course_id` to maintain per-course context.
    # Calculate the total number of interactions per student-course.
    activity_counts_per_course = (
        logs_non_graded.groupby(["student_id", "course_id"])["task_id"]
        .count()
        .reset_index(name="non_graded_interactions_per_course")
    )
    
    # Calculate the number of unique tasks viewed per student-course.
    task_coverage_per_course = (
        logs_non_graded.groupby(["student_id", "course_id"])["task_id"]
        .nunique()
        .reset_index(name="unique_non_graded_tasks_viewed_per_course")
    )

    # 4. Merge the two new feature DataFrames.
    features_per_course = activity_counts_per_course.merge(
        task_coverage_per_course, on=["student_id", "course_id"], how="outer"
    )

    # 5. Fill NaNs with 0 for student-course pairs with no non-graded activity.
    features_per_course = features_per_course.fillna(0)

    # 6. Normalization is best done on the final training dataset.
    # Scaling these features now would be incorrect because the values are context-dependent.
    # E.g., a count of 50 non-graded interactions in a course with 100 tasks is different from 50 interactions in a course with 51 tasks.
    # The normalization should happen after all features are created and the training set is finalized.

    return features_per_course

# Example of how to use this revised function and merge the data
# First, you need a DataFrame that combines students and courses for which you want to predict a label.
# This DataFrame should be your main training data structure.
# For example, you can create a DataFrame of all unique (student_id, course_id) pairs from your labels.
df_train_base = df_labels[['student_id', 'course_id']].drop_duplicates()

# Now, generate the new features.
student_course_non_graded_features = course_task_non_grade_revised(df_courses_tasks, df_activity_log)

# Merge the student features (age, gender, etc.) with the base training data.
df_train = df_train_base.merge(df_pp_students, on='student_id', how='left')

# Merge the per-course non-graded features.
df_train = df_train.merge(
    student_course_non_graded_features, 
    on=['student_id', 'course_id'], 
    how='left'
)

# Fill NaNs for students who had no non-graded interactions in specific courses.
df_train[['non_graded_interactions_per_course', 'unique_non_graded_tasks_viewed_per_course']] = df_train[
    ['non_graded_interactions_per_course', 'unique_non_graded_tasks_viewed_per_course']
].fillna(0)


In [4]:
# NEW - Course-level difficulty
def compute_difficulty_revised(df_labels):
    """
    Computes course difficulty based on the average final mark for each course.
    Lower average mark indicates a higher difficulty.
    
    Args:
        df_labels (pd.DataFrame): DataFrame with 'course_id' and 'final_mark' for each student.
        
    Returns:
        pd.DataFrame: A DataFrame with 'course_id' and 'course_difficulty'.
    """
    # 1. Group by course and compute the average final mark for each course.
    course_avg_mark = df_labels.groupby("course_id")["final_mark"].mean().reset_index(name="avg_final_mark")
    
    # 2. Compute course difficulty as the inverse of the normalized average final mark.
    # We use a simple normalization to scale the difficulty from 0 to 1.
    # The lowest average mark will get a difficulty of 1, and the highest will get a difficulty of 0.
    min_mark = course_avg_mark["avg_final_mark"].min()
    max_mark = course_avg_mark["avg_final_mark"].max()
    
    if max_mark == min_mark:
        # Avoid division by zero if all courses have the same average mark.
        course_avg_mark["normalized_mark"] = 0
    else:
        course_avg_mark["normalized_mark"] = (course_avg_mark["avg_final_mark"] - min_mark) / (max_mark - min_mark)

    # 3. The difficulty is 1 minus the normalized mark.
    course_avg_mark["course_difficulty"] = 1 - course_avg_mark["normalized_mark"]
    
    return course_avg_mark[["course_id", "course_difficulty"]]

# Example usage with your labels
# Assuming you have a df_labels DataFrame with 'student_id', 'course_id', and 'final_mark'
# Note: You need to use your labels dataset for this, as it contains the final_mark.
# The `df_task_marks` you provided doesn't have the final_mark, but the labels dataset does.
# This revised function relies on the final outcome for a more accurate difficulty metric.
# Let's assume you have a df_labels DataFrame

course_difficulty = compute_difficulty_revised(df_labels)

# You can then merge this with your main training data on `course_id`.
df_train = df_train.merge(course_difficulty, on="course_id", how="left")

In [ ]:
def get_past_performance(df_labels):
    """
    Computes a student's average past final mark and completion rate.
    """
    # Sort by date to process chronologically (assuming `final_mark` implies course completion)
    df_labels = df_labels.sort_values(by="course_id") # Assuming course_id is chronological

    student_data = []
    
    # Calculate past performance for each student and course
    for (student_id, course_id), group in df_labels.groupby(["student_id", "course_id"]):
        past_marks = df_labels[
            (df_labels["student_id"] == student_id) & (df_labels["course_id"] < course_id)
        ]["final_mark"]
        
        # Calculate average past mark
        avg_past_mark = past_marks.mean() if not past_marks.empty else 0
        
        # Calculate completion rate (assuming a mark > 0 is a completion)
        completion_rate = (past_marks > 0).mean() if not past_marks.empty else 0
        
        student_data.append({
            "student_id": student_id,
            "course_id": course_id,
            "average_past_final_mark": avg_past_mark,
            "completion_rate": completion_rate
        })
    
    return pd.DataFrame(student_data)
test = get_past_performance(df_labels)


In [12]:
test

,student_id,course_id,average_past_final_mark,completion_rate
0,STU001C1A,CRS873520,0.0,0.0
1,STU004FF9,CRS283DC7,0.0,0.0
2,STU004FF9,CRS7BD731,0.0,0.0
3,STU00641A,CRSCCEF31,0.0,0.0
4,STU0065A7,CRS675703,0.0,0.0
...,...,...,...,...
2955,STUFFDB9D,CRS94D1D4,0.0,0.0
2956,STUFFE655,CRS7A2125,0.0,0.0
2957,STUFFF26A,CRS9A7763,0.0,0.0
2958,STUFFF768,CRS6F5FBE,0.0,0.0


In [5]:
df_train["workload_x_course_difficulty"] = (
    df_train["non_graded_interactions_per_course"] * df_train["course_difficulty"]
)
ects_df = df_courses_tasks[['course_id', 'course_ects']].drop_duplicates()

df_train["final_mark_rounded"] = df_labels["final_mark"].apply(lambda x: int(np.round(x / 10) * 10))
# Now, merge this with your main training dataframe
train_df = df_train.merge(ects_df, on='course_id', how='left')
train_df.dropna(inplace=True)

In [6]:
X = train_df.drop(columns=["student_id", "course_id", "final_mark_rounded"])

# Encode target as 0-10 (11 classes)
y = train_df["final_mark_rounded"]

# Split train/validation
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
def optimize_model_cls(trial, model_name, X_train, y_train, X_val, y_val):
    """
    Optimizes classification models using Optuna.
    Returns negative F1-score (for minimization).
    """
    
    if model_name == "RandomForest":
        n_estimators = trial.suggest_int("n_estimators", 100, 1000)
        max_depth = trial.suggest_int("max_depth", 3, 55)
        min_samples_split = trial.suggest_int("min_samples_split", 1, 50)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 50)
        max_features = trial.suggest_categorical("max_features", ["sqrt", "log2", None])
        
        model = RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            max_features=max_features,
            random_state=42,
            n_jobs=-1
        )
    # Train
    model.fit(X_train, y_train)
    
    # Predict
    preds = model.predict(X_val)
    
    # Calculate metrics
    acc = accuracy_score(y_val, preds)
    f1 = f1_score(y_val, preds, average="weighted", zero_division=0)
    recall = recall_score(y_val, preds, average="weighted", zero_division=0)
    precision = precision_score(y_val, preds, average="weighted", zero_division=0)
    
    # Log metrics for Optuna trial
    trial.set_user_attr("accuracy", acc)
    trial.set_user_attr("f1", f1)
    trial.set_user_attr("recall", recall)
    trial.set_user_attr("precision", precision)
    
    # Objective: maximize F1-score (minimize negative F1)
    return -f1


# Main optimization loop
models_to_optimize = ["RandomForest"]
best_models = {}

for model_name in models_to_optimize:
    print(f"\n{'='*50}")
    print(f"Optimizing {model_name}...")
    print(f"{'='*50}")
    
    study = optuna.create_study(direction="minimize")
    study.optimize(
        lambda trial: optimize_model_cls(trial, model_name, X_train, y_train, X_val, y_val),
        n_trials=100,  # Increased for better optimization
        show_progress_bar=True
    )
    
    best_models[model_name] = {
        "best_params": study.best_params,
        "best_f1": study.best_trial.user_attrs.get("f1"),
        "best_accuracy": study.best_trial.user_attrs.get("accuracy"),
        "best_recall": study.best_trial.user_attrs.get("recall"),
        "best_precision": study.best_trial.user_attrs.get("precision")
    }
    
    print(f"\nBest F1-score for {model_name}: {best_models[model_name]['best_f1']:.4f}")
    print(f"Accuracy: {best_models[model_name]['best_accuracy']:.4f}")
    print(f"Recall: {best_models[model_name]['best_recall']:.4f}")
    print(f"Precision: {best_models[model_name]['best_precision']:.4f}")
    print(f"Best params: {study.best_params}")


[I 2025-10-03 14:19:54,154] A new study created in memory with name: no-name-a3b0719c-f03d-41d0-b3cb-1a51f8d9bdc6



Optimizing RandomForest...


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2025-10-03 14:19:54,544] Trial 0 finished with value: -0.10598357884727894 and parameters: {'n_estimators': 258, 'max_depth': 14, 'min_samples_split': 7, 'min_samples_leaf': 5, 'max_features': None}. Best is trial 0 with value: -0.10598357884727894.
[I 2025-10-03 14:19:54,970] Trial 1 finished with value: -0.10780909295761915 and parameters: {'n_estimators': 282, 'max_depth': 18, 'min_samples_split': 7, 'min_samples_leaf': 2, 'max_features': None}. Best is trial 1 with value: -0.10780909295761915.
[I 2025-10-03 14:19:55,320] Trial 2 finished with value: -0.08691265133610221 and parameters: {'n_estimators': 279, 'max_depth': 5, 'min_samples_split': 5, 'min_samples_leaf': 2, 'max_features': 'log2'}. Best is trial 1 with value: -0.10780909295761915.
[I 2025-10-03 14:19:55,543] Trial 3 finished with value: -0.09599748149181933 and parameters: {'n_estimators': 170, 'max_depth': 20, 'min_samples_split': 4, 'min_samples_leaf': 2, 'max_features': 'sqrt'}. Best is trial 1 with value: -0.1078

[I 2025-10-03 14:20:04,894] A new study created in memory with name: no-name-31834595-b046-4abc-b11a-0fe600387d6d


[I 2025-10-03 14:20:04,893] Trial 49 finished with value: -0.08060953972823635 and parameters: {'n_estimators': 216, 'max_depth': 4, 'min_samples_split': 5, 'min_samples_leaf': 1, 'max_features': 'log2'}. Best is trial 42 with value: -0.12406301780285646.

Best F1-score for RandomForest: 0.1241
Accuracy: 0.1554
Recall: 0.1554
Precision: 0.1245
Best params: {'n_estimators': 100, 'max_depth': 7, 'min_samples_split': 7, 'min_samples_leaf': 2, 'max_features': None}

Optimizing XGBoost...


  0%|          | 0/50 [00:00<?, ?it/s]

[W 2025-10-03 14:20:04,900] Trial 0 failed with parameters: {'n_estimators': 251, 'max_depth': 9, 'learning_rate': 0.09440985958430317, 'subsample': 0.749938290900573, 'colsample_bytree': 0.7634642756496642, 'min_child_weight': 4, 'gamma': 4.25384977675782} because of the following error: NameError("name 'XGBClassifier' is not defined").
Traceback (most recent call last):
  File "c:\Users\Anurath\AppData\Local\Programs\Python\Python313\Lib\site-packages\optuna\study\_optimize.py", line 201, in _run_trial
    value_or_values = func(trial)
  File "C:\Users\Anurath\AppData\Local\Temp\ipykernel_11376\86162981.py", line 108, in <lambda>
    lambda trial: optimize_model_cls(trial, model_name, X_train, y_train, X_val, y_val),
                  ~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Anurath\AppData\Local\Temp\ipykernel_11376\86162981.py", line 33, in optimize_model_cls
    model = XGBClassifier(
            ^^^^^^^^^^^^^
NameError: name 'XGBClass

NameError: name 'XGBClassifier' is not defined